In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip -q install ultralytics

import os, shutil
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm import tqdm

from ultralytics import YOLO


In [ ]:
import pandas as pd

train_df = pd.read_csv("../input/m-2-big-data-dlia-project-2025-clean-dataset/train.csv")

print("Columns:")
print(train_df.columns)

print("\nSample rows:")
train_df.head()


In [ ]:
print(train_df[["x_min","y_min","x_max","y_max"]].describe())

In [ ]:
assert (train_df["x_max"] > train_df["x_min"]).all()
assert (train_df["y_max"] > train_df["y_min"]).all()
print("✅ All boxes have valid geometry")

In [ ]:
from PIL import Image
import os

IMAGES_DIR = "../input/m-2-big-data-dlia-project-2025-clean-dataset/train"

sizes = set()
for img_name in train_df["image_id"].unique()[:20]:
    img = Image.open(os.path.join(IMAGES_DIR, img_name))
    sizes.add(img.size)

print("Image sizes:", sizes)

In [ ]:
# train_df must contain:
# ['image_id','label','x_min','y_min','x_max','y_max']

print(train_df.columns)
print("n images:", train_df["image_id"].nunique())
print("labels:", sorted(train_df["label"].unique()))


In [ ]:
import matplotlib.pyplot as plt
import random

row = train_df.sample(1).iloc[0]

img = Image.open(os.path.join(IMAGES_DIR, row["image_id"]))
plt.figure(figsize=(6,6))
plt.imshow(img)
plt.axis("off")

x1, y1, x2, y2 = row["x_min"], row["y_min"], row["x_max"], row["y_max"]
print(x1, y1, x2, y2)
plt.gca().add_patch(
    plt.Rectangle(
        (x1, y1),
        x2 - x1,
        y2 - y1,
        fill=False,
        edgecolor="red",
        linewidth=2
    )
)


plt.title(row["label"])
plt.show()

In [ ]:
print(train_df["label"].value_counts())


In [ ]:
labels = sorted(train_df["label"].unique())
label2id = {l:i for i,l in enumerate(labels)}
id2label = {i:l for l,i in label2id.items()}

NUM_CLASSES = len(labels)
print("NUM_CLASSES:", NUM_CLASSES)
print(label2id)


In [ ]:
IMAGES_DIR = "../input/m-2-big-data-dlia-project-2025-clean-dataset/train"

YOLO_ROOT = "yolo8cls"
IMG_TRAIN = f"{YOLO_ROOT}/images/train"
IMG_VAL   = f"{YOLO_ROOT}/images/val"
LBL_TRAIN = f"{YOLO_ROOT}/labels/train"
LBL_VAL   = f"{YOLO_ROOT}/labels/val"

for p in [IMG_TRAIN, IMG_VAL, LBL_TRAIN, LBL_VAL]:
    os.makedirs(p, exist_ok=True)

# split by image_id (important: not by bbox row)
image_ids = train_df["image_id"].unique()
train_ids, val_ids = train_test_split(image_ids, test_size=0.2, random_state=42)

train_ids = set(train_ids)
val_ids   = set(val_ids)

def write_yolo_labels(image_id, subset):
    img_path = os.path.join(IMAGES_DIR, image_id)
    img = Image.open(img_path)
    W, H = img.size

    rows = train_df[train_df["image_id"] == image_id]

    img_out = IMG_TRAIN if subset == "train" else IMG_VAL
    lbl_out = LBL_TRAIN if subset == "train" else LBL_VAL

    # copy image
    shutil.copy(img_path, os.path.join(img_out, image_id))

    # write label file
    label_file = os.path.join(lbl_out, image_id.replace(".png", ".txt"))
    with open(label_file, "w") as f:
        for _, r in rows.iterrows():
            x1, y1, x2, y2 = float(r["x_min"]), float(r["y_min"]), float(r["x_max"]), float(r["y_max"])
            cls_id = label2id[r["label"]]

            # xyxy -> normalized xywh
            xc = ((x1 + x2) / 2.0) / W
            yc = ((y1 + y2) / 2.0) / H
            bw = (x2 - x1) / W
            bh = (y2 - y1) / H

            # clamp for safety
            xc = min(max(xc, 0.0), 1.0)
            yc = min(max(yc, 0.0), 1.0)
            bw = min(max(bw, 0.0), 1.0)
            bh = min(max(bh, 0.0), 1.0)

            # skip degenerate boxes
            if bw <= 0 or bh <= 0:
                continue

            f.write(f"{cls_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")

print("Converting YOLO train...")
for img_id in tqdm(list(train_ids)[:]):  # all
    write_yolo_labels(img_id, "train")

print("Converting YOLO val...")
for img_id in tqdm(list(val_ids)[:]):    # all
    write_yolo_labels(img_id, "val")

print("✅ YOLO multi-class dataset ready")

In [ ]:
yaml_path = f"{YOLO_ROOT}.yaml"

with open(yaml_path, "w") as f:
    f.write(f"path: {YOLO_ROOT}\n")
    f.write("train: images/train\n")
    f.write("val: images/val\n")
    f.write(f"nc: {NUM_CLASSES}\n")
    f.write("names:\n")
    for i in range(NUM_CLASSES):
        f.write(f"  {i}: {id2label[i]}\n")

print("✅ YAML written:", yaml_path)
print(open(yaml_path).read())


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8l.pt")  # ⬅️ IMPORTANT

model.train(
    data=yaml_path,
    epochs=120,
    imgsz=1024,           # ⬅️ crucial pour radiographies
    batch=4,              # GPU P100 ok
    lr0=0.003,
    cos_lr=True,
    box=7.5,
    cls=1.0,              # ⬅️ IMPORTANT (pas 0.0)
    patience=20,
    mosaic=0.2,
    mixup=0.0,
    device=0
)


In [ ]:
!pip install -q ensemble-boxes

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from PIL import Image
from ultralytics import YOLO
from ensemble_boxes import weighted_boxes_fusion

# --------------------------------------------------
# CONFIG
# --------------------------------------------------
TEST_IMG_DIR = "../input/m-2-big-data-dlia-project-2025-clean-dataset/test"
mapping_df = pd.read_csv(
    "../input/m-2-big-data-dlia-project-2025-clean-dataset/ID_to_Image_Mapping.csv"
)

image_to_ID = dict(zip(mapping_df["image_id"], mapping_df["ID"]))

CONF_TTA = 0.01
IOU_WBF = 0.5

# --------------------------------------------------
# LOAD YOLO
# --------------------------------------------------
best_path = "runs/detect/train/weights/best.pt"
yolo = YOLO(best_path)

# --------------------------------------------------
# WBF helper
# --------------------------------------------------
def apply_wbf(boxes, scores, labels, W, H):
    if len(boxes) == 0:
        return []

    boxes_n = [[
        [b[0]/W, b[1]/H, b[2]/W, b[3]/H] for b in boxes
    ]]
    scores_n = [scores]
    labels_n = [labels]

    fused_boxes, fused_scores, fused_labels = weighted_boxes_fusion(
        boxes_n,
        scores_n,
        labels_n,
        iou_thr=IOU_WBF,
        skip_box_thr=0.001
    )

    final = []
    for b,s,l in zip(fused_boxes, fused_scores, fused_labels):
        final.append((
            b[0]*W, b[1]*H, b[2]*W, b[3]*H,
            float(s),
            int(l)
        ))
    return final

# --------------------------------------------------
# TTA prediction
# --------------------------------------------------
def predict_tta(yolo, img_path):
    preds = []

    # normal
    r1 = yolo.predict(
        img_path, imgsz=1024, conf=CONF_TTA, device=0, verbose=False
    )[0]

    # horizontal flip
    r2 = yolo.predict(
        img_path, imgsz=1024, conf=CONF_TTA, augment=True, device=0, verbose=False
    )[0]

    for r in [r1, r2]:
        if r.boxes is None:
            continue
        for b, s, c in zip(
            r.boxes.xyxy.cpu().numpy(),
            r.boxes.conf.cpu().numpy(),
            r.boxes.cls.cpu().numpy().astype(int)
        ):
            preds.append((b, float(s), int(c)))

    return preds

# --------------------------------------------------
# INFERENCE LOOP
# --------------------------------------------------
rows = []

for image_id in sorted(os.listdir(TEST_IMG_DIR)):
    if image_id not in image_to_ID:
        continue

    ID = int(image_to_ID[image_id])
    img_path = os.path.join(TEST_IMG_DIR, image_id)

    img = Image.open(img_path).convert("RGB")
    W, H = img.size

    preds = predict_tta(yolo, img_path)

    if len(preds) == 0:
        rows.append([ID, image_id, 0.0, 0.0, float(W), float(H), 1e-6, labels[0]])
        continue

    boxes = [p[0].tolist() for p in preds]
    scores = [p[1] for p in preds]
    clsids = [p[2] for p in preds]

    fused = apply_wbf(boxes, scores, clsids, W, H)

    if len(fused) == 0:
        rows.append([ID, image_id, 0.0, 0.0, float(W), float(H), 1e-6, labels[0]])
        continue

    # 🔥 choisir la meilleure box fusionnée
    fused = sorted(fused, key=lambda x: x[4], reverse=True)
    x1,y1,x2,y2,score,cls_id = fused[0]

    # recalibration confidence
    final_conf = score ** 0.7

    label = id2label[cls_id]

    rows.append([
        ID,
        image_id,
        float(x1), float(y1),
        float(x2), float(y2),
        float(final_conf),
        label
    ])

# --------------------------------------------------
# SUBMISSION
# --------------------------------------------------
submission_df = pd.DataFrame(
    rows,
    columns=["ID","image_id","x_min","y_min","x_max","y_max","confidence","label"]
)

# HARD CHECKS
assert submission_df["ID"].is_unique
assert set(submission_df["ID"]) == set(mapping_df["ID"])
assert submission_df["confidence"].between(0,1).all()
assert (submission_df["x_max"] > submission_df["x_min"]).all()
assert (submission_df["y_max"] > submission_df["y_min"]).all()

submission_df.to_csv("submission.csv", index=False)
print("✅ submission.csv saved | rows:", len(submission_df))
submission_df.head()


In [ ]:
import os
import torch
import pandas as pd
import numpy as np

from PIL import Image
from tqdm import tqdm

import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.transforms import functional as F

from torch.utils.data import Dataset, DataLoader

In [ ]:
IMAGES_DIR = "../input/m-2-big-data-dlia-project-2025-clean-dataset/train"

class ChestXrayDetectionDataset(Dataset):
    def __init__(self, df, transforms=None):
        self.df = df
        self.image_ids = df["image_id"].unique()
        self.transforms = transforms

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        rows = self.df[self.df["image_id"] == image_id]

        img_path = os.path.join(IMAGES_DIR, image_id)
        img = Image.open(img_path).convert("RGB")
        W, H = img.size

        boxes = []
        labels = []

        for _, r in rows.iterrows():
            x1,y1,x2,y2 = r["x_min"], r["y_min"], r["x_max"], r["y_max"]

            # clamp
            x1,y1 = max(0,x1), max(0,y1)
            x2,y2 = min(W,x2), min(H,y2)

            if x2 <= x1 or y2 <= y1:
                continue

            boxes.append([x1,y1,x2,y2])
            labels.append(label2id[r["label"]])

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx])
        }

        if self.transforms:
            img = self.transforms(img)

        return img, target


In [ ]:
def get_transforms(train=True):
    if train:
        return torchvision.transforms.Compose([
            torchvision.transforms.ToTensor(),
            torchvision.transforms.RandomHorizontalFlip(0.5),
        ])
    else:
        return torchvision.transforms.Compose([
            torchvision.transforms.ToTensor(),
        ])

from sklearn.model_selection import train_test_split

train_ids, val_ids = train_test_split(
    train_df["image_id"].unique(),
    test_size=0.2,
    random_state=42
)

train_df_frcnn = train_df[train_df["image_id"].isin(train_ids)]
val_df_frcnn   = train_df[train_df["image_id"].isin(val_ids)]

train_ds = ChestXrayDetectionDataset(train_df_frcnn, get_transforms(train=True))
val_ds   = ChestXrayDetectionDataset(val_df_frcnn, get_transforms(train=False))

def collate_fn(batch):
    return tuple(zip(*batch))

train_loader = DataLoader(
    train_ds, batch_size=4, shuffle=True,
    collate_fn=collate_fn, num_workers=2
)

val_loader = DataLoader(
    val_ds, batch_size=2, shuffle=False,
    collate_fn=collate_fn, num_workers=2
)


In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

NUM_CLASSES = 9  # 8 pathologies + background

model = fasterrcnn_resnet50_fpn(weights="DEFAULT")

# Remplacer la tête
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)

# 🔒 Geler le backbone partiellement
for name, param in model.backbone.body.named_parameters():
    if "layer3" not in name and "layer4" not in name:
        param.requires_grad = False

model.to(device)
print("✅ Model ready (partial backbone frozen)")

In [ ]:
params = [p for p in model.parameters() if p.requires_grad]

optimizer = torch.optim.SGD(
    params,
    lr=0.005,          # ⬅️ PLUS GRAND
    momentum=0.9,
    weight_decay=1e-4
)

In [ ]:
NUM_EPOCHS = 15

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0

    for images, targets in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        images = [img.to(device) for img in images]
        targets = [{k:v.to(device) for k,v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch+1} | Train loss: {avg_loss:.4f}")


In [ ]:
model.eval()
with torch.no_grad():
    img_id = train_df["image_id"].iloc[0]
    img = Image.open(os.path.join(IMAGES_DIR, img_id)).convert("RGB")
    img_tensor = F.to_tensor(img).to(device)

    out = model([img_tensor])[0]

print("Boxes:", len(out["boxes"]))
print("Scores:", out["scores"][:5])
print("Labels:", out["labels"][:5])


In [ ]:
TEST_IMG_DIR = "../input/m-2-big-data-dlia-project-2025-clean-dataset/test"

image_to_ID = dict(zip(mapping_df["image_id"], mapping_df["ID"]))

model.eval()
rows = []

with torch.no_grad():
    for image_id in tqdm(mapping_df["image_id"]):
        img_path = os.path.join(TEST_IMG_DIR, image_id)
        img = Image.open(img_path).convert("RGB")
        W,H = img.size

        img_tensor = F.to_tensor(img).to(device)
        outputs = model([img_tensor])[0]

        if len(outputs["boxes"]) == 0:
            rows.append([image_to_ID[image_id], image_id, 0,0,W,H,1e-6, labels[0]])
            continue

        scores = outputs["scores"].cpu().numpy()
        boxes  = outputs["boxes"].cpu().numpy()
        clsids = outputs["labels"].cpu().numpy()

        best = scores.argmax()

        x1,y1,x2,y2 = boxes[best]
        conf = float(scores[best])
        lab  = id2label[int(clsids[best])]

        rows.append([
            int(image_to_ID[image_id]),
            image_id,
            float(x1), float(y1), float(x2), float(y2),
            conf,
            lab
        ])

submission_df = pd.DataFrame(
    rows,
    columns=["ID","image_id","x_min","y_min","x_max","y_max","confidence","label"]
)

assert submission_df["ID"].is_unique
submission_df.to_csv("submission_frcnn.csv", index=False)

print("✅ Faster R-CNN submission saved")


In [ ]:
# Sauvegarde du modèle Faster R-CNN
torch.save(
    model.state_dict(),
    "fasterrcnn_best.pth"
)

print("✅ Faster R-CNN checkpoint saved")

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm import tqdm

def iou_xyxy(a, b):
    # a,b: [x1,y1,x2,y2]
    x1 = max(a[0], b[0]); y1 = max(a[1], b[1])
    x2 = min(a[2], b[2]); y2 = min(a[3], b[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    area_a = max(0, a[2]-a[0]) * max(0, a[3]-a[1])
    area_b = max(0, b[2]-b[0]) * max(0, b[3]-b[1])
    union = area_a + area_b - inter + 1e-9
    return inter / union

def nms(cands, iou_thr=0.5):
    """
    cands: list of dict {box, score, label, src}
    returns filtered list sorted by score desc
    """
    cands = sorted(cands, key=lambda d: d["score"], reverse=True)
    keep = []
    for c in cands:
        ok = True
        for k in keep:
            if iou_xyxy(c["box"], k["box"]) >= iou_thr:
                ok = False
                break
        if ok:
            keep.append(c)
    return keep

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:
TEST_IMG_DIR = "../input/m-2-big-data-dlia-project-2025-clean-dataset/test"

# mapping_df must contain columns ["ID","image_id"]
# Exemple:
# mapping_df = pd.read_csv("../input/.../mapping.csv")

assert "ID" in mapping_df.columns and "image_id" in mapping_df.columns

image_to_ID = dict(zip(mapping_df["image_id"], mapping_df["ID"]))
ID_to_image = dict(zip(mapping_df["ID"], mapping_df["image_id"]))

print("✅ mapping loaded | test images:", len(os.listdir(TEST_IMG_DIR)))


In [ ]:
from ultralytics import YOLO
yolo = YOLO("runs/detect/train/weights/best.pt")
print("✅ YOLO loaded")

In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import torchvision.transforms.functional as TF

NUM_CLASSES = 9  # 8 classes + background

frcnn = fasterrcnn_resnet50_fpn(weights=None)
in_features = frcnn.roi_heads.box_predictor.cls_score.in_features
frcnn.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)

ckpt = torch.load("fasterrcnn_best.pth", map_location=device)  # <-- adapte si besoin
frcnn.load_state_dict(ckpt)
frcnn.to(device).eval()

print("✅ Faster R-CNN loaded")

In [ ]:
# ⚠️ Mets EXACTEMENT l'ordre utilisé par YOLO training (yolo_boxes.yaml -> names)
yolo_names = [
    "Atelectasis",
    "Cardiomegaly",
    "Effusion",
    "Infiltrate",
    "Mass",
    "Nodule",
    "Pneumonia",
    "Pneumothorax"
]

id2label_yolo = {i: n for i, n in enumerate(yolo_names)}
label2id_yolo = {n: i for i, n in enumerate(yolo_names)}

# FasterRCNN outputs labels in [1..8] (0 = background)
id2label_frcnn = {i+1: n for i, n in enumerate(yolo_names)}  # 1..8
label2id_frcnn = {n: i+1 for i, n in enumerate(yolo_names)}

print("✅ labels ready")

In [ ]:
CONF_YOLO = 0.03
CONF_FRCNN = 0.03
IOU_NMS = 0.5

submission_rows = []

test_files = sorted(os.listdir(TEST_IMG_DIR))

for image_id in tqdm(test_files):
    if image_id not in image_to_ID:
        continue

    ID = int(image_to_ID[image_id])
    img_path = os.path.join(TEST_IMG_DIR, image_id)

    img = Image.open(img_path).convert("RGB")
    W, H = img.size

    cands = []

    # -------------------
    # 1) YOLO candidates
    # -------------------
    yres = yolo.predict(source=img_path, imgsz=1024, conf=CONF_YOLO, iou=0.6, device=0, verbose=False)[0]
    if yres.boxes is not None and len(yres.boxes) > 0:
        boxes = yres.boxes.xyxy.cpu().numpy()
        scores = yres.boxes.conf.cpu().numpy()
        clsids = yres.boxes.cls.cpu().numpy().astype(int)

        for b, s, c in zip(boxes, scores, clsids):
            x1,y1,x2,y2 = b.tolist()
            x1 = float(max(0, min(W, x1))); y1 = float(max(0, min(H, y1)))
            x2 = float(max(0, min(W, x2))); y2 = float(max(0, min(H, y2)))
            if x2 <= x1 or y2 <= y1: 
                continue
            cands.append({
                "box": [x1,y1,x2,y2],
                "score": float(s),
                "label": id2label_yolo[int(c)],
                "src": "yolo"
            })

    # -------------------------
    # 2) FasterRCNN candidates
    # -------------------------
    with torch.no_grad():
        x = TF.to_tensor(img).to(device)  # 3 channels guaranteed by RGB
        out = frcnn([x])[0]

    if out is not None and len(out["boxes"]) > 0:
        boxes = out["boxes"].detach().cpu().numpy()
        scores = out["scores"].detach().cpu().numpy()
        labels = out["labels"].detach().cpu().numpy().astype(int)

        for b, s, lab_id in zip(boxes, scores, labels):
            if s < CONF_FRCNN:
                continue
            if lab_id == 0: 
                continue
            x1,y1,x2,y2 = b.tolist()
            x1 = float(max(0, min(W, x1))); y1 = float(max(0, min(H, y1)))
            x2 = float(max(0, min(W, x2))); y2 = float(max(0, min(H, y2)))
            if x2 <= x1 or y2 <= y1:
                continue
            cands.append({
                "box": [x1,y1,x2,y2],
                "score": float(s),
                "label": id2label_frcnn[int(lab_id)],
                "src": "frcnn"
            })

    # -------------------------
    # 3) Ensemble: NMS then pick best
    # -------------------------
    if len(cands) == 0:
        # fallback (1 row per image)
        submission_rows.append([ID, image_id, 0.0, 0.0, float(W), float(H), 1e-6, yolo_names[0]])
        continue

    # Option: NMS global (toutes classes), simple et robuste pour 1 box final
    kept = nms(cands, iou_thr=IOU_NMS)

    best = kept[0]
    x1,y1,x2,y2 = best["box"]
    conf = best["score"]
    lab = best["label"]

    submission_rows.append([ID, image_id, x1,y1,x2,y2, float(conf), lab])

In [ ]:
submission_df = pd.DataFrame(
    submission_rows,
    columns=["ID","image_id","x_min","y_min","x_max","y_max","confidence","label"]
)

# HARD checks
assert submission_df["ID"].is_unique, "❌ Duplicate ID values found (must be 1 row per ID)"
assert set(submission_df["ID"]) == set(mapping_df["ID"]), "❌ IDs mismatch"
assert submission_df["confidence"].between(0,1).all()
assert (submission_df["x_max"] > submission_df["x_min"]).all()
assert (submission_df["y_max"] > submission_df["y_min"]).all()

submission_df.to_csv("submission_both.csv", index=False)
print("✅ submission_both.csv saved | rows:", len(submission_df))
submission_df.head()